# Pre Processing Images

In [ ]:
import numpy as np
from skimage.feature import hog
from skimage import exposure
from skimage import color

import matplotlib.pyplot as plt
import cv2
from sklearn.metrics import accuracy_score


from datetime import datetime
from sklearn.utils import check_random_state
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split

## Loading and labelling the images

In [ ]:
cloud_heights = ["Low", "Medium", "High"]

# cloud_heights = ["Low", "Medium-High"]

high_lvl_clouds = ["Ci", "Cc", "Cs", "Ct"]

mid_lvl_clouds = ["Ac", "As", "Ns"]

low_lvl_clouds = ["Cu", "Cb", "Sc", "St"]

In [ ]:
import os
# identifyign folders/labels
cloud_labels = os.listdir("../resources/cloud-images/CCSN_v2")
# Removing the .DS_Store file - autogenerated metadata for a folder - native to macOS
if ".DS_Store" in cloud_labels: cloud_labels.remove(".DS_Store")

labeled_data = {}

for label in cloud_labels:
    cloud_images = os.listdir("../resources/cloud-images/CCSN_v2/" + label)
    if label in high_lvl_clouds:
        for image in cloud_images:
            labeled_data[image] =  [{'label': label, 'height': "High"}]

    if label in mid_lvl_clouds:
        for image in cloud_images:
            labeled_data[image] =  [{'label': label, 'height': "Medium"}]

    if label in low_lvl_clouds:
        for image in cloud_images:
            labeled_data[image] =  [{'label': label, 'height': "Low" }]

print(labeled_data)

## Preprocessing the Images

In [ ]:

train_images = []
train_heights = []

start_datetime = datetime.now()

for (i, image_file) in enumerate(labeled_data):
    #read image
    path = '../resources/cloud-images/CCSN_v2/'+ labeled_data[image_file][0]['label'] + '/' + image_file
    image = cv2.imread(path)
    # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
    # image = color.rgb2gray(image)
    image = cv2.resize(image, (128, 128))
    # print(image)
    image = np.array(image)
    fd, hog_image = hog(
        image,
        orientations=16,
        pixels_per_cell=(4, 4),
        cells_per_block=(2, 2),
        visualize=True,
        channel_axis=-1,
    )

    print(fd.shape)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)

    ax1.axis('off')
    ax1.imshow(image, cmap=plt.cm.gray)
    ax1.set_title('Input image')

    # Rescale histogram for better display
    hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))

    ax2.axis('off')
    ax2.imshow(hog_image_rescaled, cmap=plt.cm.gray)
    ax2.set_title('Histogram of Oriented Gradients')

    height = cloud_heights.index(labeled_data[image_file][0]['height'])
    tmp_height = labeled_data[image_file][0]['height']

    train_images.append(fd)
    train_heights.append(height)
    print('Loaded...', '\U0001F483', 'Image', str(i+1), 'is a', tmp_height)
    plt.show()

end_datetime = datetime.now()

In [ ]:
train_images = np.array(train_images)
train_heights = np.array(train_heights)

X, y = train_images,train_heights

In [ ]:
random_state = check_random_state(0)
permutation = random_state.permutation(X.shape[0])
X = X[permutation]
y = y[permutation]
X = X.reshape((X.shape[0], -1))

print(len(X))

In [ ]:
print('Image Processing Duration: ' + str(end_datetime-start_datetime))

# Splitting the data

test_size = 0.3

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=None)

In [ ]:
rf_classifier = RandomForestClassifier()
print(X_train.shape)


In [ ]:
rf_classifier.fit(X_train, y_train)


In [ ]:
y_pred_rf = rf_classifier.predict(X_test)

In [ ]:
print("Accuracy: "+str(accuracy_score(y_test, y_pred_rf)))